# Inventário reprodutível dos datasets locais

Este notebook refaz o inventário que sustenta o relatório de datasets. Ele não baixa dados, não treina modelos e não inicializa TensorFlow/GPU. A coleta de metadados de arquivos é executada pelo Windows para evitar a lentidão de enumeração de `/mnt/c` no WSL; os adaptadores Python do benchmark validam rótulos, splits e arrays.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT = PROJECT_ROOT / 'outputs' / 'dataset-inventory-2026-07-28'
OUTPUT.mkdir(parents=True, exist_ok=True)

def windows_path(path: Path) -> str:
    return subprocess.check_output(['wslpath', '-w', str(path)], text=True).strip()

def run(command: list[str]) -> None:
    print('$', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

run([
    'powershell.exe', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File',
    windows_path(PROJECT_ROOT / 'scripts' / 'profile_dataset_disk.ps1'),
    '-Output', windows_path(OUTPUT / 'disk_profile.json'),
])
run([
    'powershell.exe', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File',
    windows_path(PROJECT_ROOT / 'scripts' / 'index_folder_datasets.ps1'),
    '-Output', windows_path(OUTPUT / 'folder_index.json'),
])
run([
    sys.executable, 'scripts/profile_datasets.py',
    '--output', str(OUTPUT),
    '--disk-profile', str(OUTPUT / 'disk_profile.json'),
    '--folder-index', str(OUTPUT / 'folder_index.json'),
    '--sample-per-class', '24',
])

profile = json.loads((OUTPUT / 'dataset_profile.json').read_text(encoding='utf-8'))
print(f"Inventario atualizado: {profile['generated_at_utc']} - {len(profile['datasets'])} datasets")

$ powershell.exe -NoProfile -ExecutionPolicy Bypass -File C:\source\repos\teste-modelos-tcc\scripts\profile_dataset_disk.ps1 -Output C:\source\repos\teste-modelos-tcc\outputs\dataset-inventory-2026-07-28\disk_profile.json


[disk] mnist
[disk] fashion_mnist
[disk] kmnist
[disk] emnist_balanced
[disk] cifar10
[disk] cifar100_coarse
[disk] cinic10


[disk] svhn
[disk] gtsrb


[disk] fer2013
Perfil de disco salvo em: C:\source\repos\teste-modelos-tcc\outputs\dataset-inventory-2026-07-28\disk_profile.json
$ powershell.exe -NoProfile -ExecutionPolicy Bypass -File C:\source\repos\teste-modelos-tcc\scripts\index_folder_datasets.ps1 -Output C:\source\repos\teste-modelos-tcc\outputs\dataset-inventory-2026-07-28\folder_index.json


[index] cinic10


...

[7/10] Perfilando cinic10...


[8/10] Perfilando svhn...


[9/10] Perfilando gtsrb...


[10/10] Perfilando fer2013...


Inventário salvo em: /mnt/c/source/repos/teste-modelos-tcc/outputs/dataset-inventory-2026-07-28
Inventario atualizado: 2026-07-28T09:22:29.372750Z - 10 datasets


In [2]:
total_images = sum(item['total_images'] for item in profile['datasets'])
total_bytes = sum(item['disk']['root_bytes'] for item in profile['datasets'])
print(f"Total de imagens: {total_images:,}")
print(f"Espaço local: {total_bytes / 1024**3:.2f} GiB")
print()
print('| Dataset | Imagens | Classes | Modalidade | Disco | Razão maior/menor classe |')
print('|---|---:|---:|---|---:|---:|')
for item in profile['datasets']:
    print(
        f"| {item['dataset']} | {item['total_images']:,} | {item['num_classes']} | "
        f"{item['native_summary']['color_mode']} | {item['disk']['root_bytes_human']} | "
        f"{item['imbalance_ratio_max_over_min']:.2f} |"
    )

Total de imagens: 906,046
Espaço local: 1.99 GiB

| Dataset | Imagens | Classes | Modalidade | Disco | Razão maior/menor classe |
|---|---:|---:|---|---:|---:|
| mnist | 70,000 | 10 | grayscale (1 canal) | 11,06 MiB | 1.25 |
| fashion_mnist | 70,000 | 10 | grayscale (1 canal) | 29,45 MiB | 1.00 |
| kmnist | 70,000 | 10 | grayscale (1 canal) | 20,50 MiB | 1.00 |
| emnist_balanced | 131,600 | 47 | grayscale (1 canal) | 31,49 MiB | 1.00 |
| cifar10 | 60,000 | 10 | RGB (3 canais após decodificação) | 177,59 MiB | 1.00 |
| cifar100_coarse | 60,000 | 20 | RGB (3 canais após decodificação) | 177,67 MiB | 1.00 |
| cinic10 | 270,000 | 10 | RGB (3 canais após decodificação) | 712,33 MiB | 1.00 |
| svhn | 99,289 | 10 | RGB (3 canais após decodificação) | 234,91 MiB | 3.03 |
| gtsrb | 39,270 | 43 | RGB (3 canais após decodificação) | 352,73 MiB | 85.20 |
| fer2013 | 35,887 | 7 | grayscale (1 canal) | 287,13 MiB | 16.43 |


## Arquivos produzidos

- `dataset_profile.json`: perfil completo por dataset, incluindo metadados de origem, propriedades nativas, escopo de amostragem e resultado da auditoria existente.
- `dataset_summary.csv`: comparação em uma linha por dataset.
- `class_distribution.csv`: contagens exatas por classe e split original.
- `disk_profile.json` e `folder_index.json`: evidências intermediárias usadas para tornar a enumeração de arquivos rápida e auditável.

## Limitações explícitas

- Dimensões/formato de datasets em arrays foram verificados integralmente; para datasets armazenados como arquivos, a inspeção de imagem usa até 24 exemplos por classe e declara esse escopo no JSON.
- A auditoria existente não executou `verify_images` nem `hash_images`; portanto, o inventário não afirma que todos os pixels foram verificados contra corrupção ou duplicata.
- As contagens referem-se aos splits locais de origem. O benchmark ainda une e redistribui os exemplos em 70%/15%/15% com estratificação para cada seed.